In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", None)

# Loading data

In [2]:
# alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v2.csv")
# weather = pd.read_csv("data/weather/weather_data_preprocessed_v3.csv")
# isw = pd.read_csv("data/isw/isw_data_preprocessed_v5.csv")
# telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

from app.db.database import Database


with Database("app/db/database.db") as db:
    alarms_rows = db.alarms.get()
    weather_rows = db.weather.get()
    isw_rows = db.isw.get()
    telegram_rows = db.telegram.get()

alarms = pd.DataFrame([dict(row) for row in alarms_rows])
weather = pd.DataFrame([dict(row) for row in weather_rows])
isw_raw = pd.DataFrame([dict(row) for row in isw_rows])
telegram = pd.DataFrame([dict(row) for row in telegram_rows])

Commiting


In [3]:
alarms.head(3)

,region_id,region,time,alarm,has_started,has_ended
0,3,Хмельницька обл.,2022-02-26 16:00:00,1,1,0
1,3,Хмельницька обл.,2022-02-26 17:00:00,1,0,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1,1,0


In [4]:
weather.head(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,0.0,0.0,82.64,-2.6,0.0,0.0,NaN,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,2022-02-24 00:00:00,Kropyvnytskyi
2,2.0,-1.1,80.51,-1.0,0.0,0.0,NaN,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,2022-02-24 00:00:00,Dnipro


In [5]:
isw_raw.head(3)

,id,date,title,url,text
0,1745,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,"February 24, 3:00 pm EST Russian President Vla..."
1,1746,2022-02-24,Ukraine Conflict Update 6,https://understandingwar.org/research/russia-u...,"Institute for the Study of War, Russia Team 9:..."
2,1743,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Kateryna Stepa..."


In [6]:
telegram.head(3)

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1,2022-02-24 03:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
2,2022-02-24 04:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0


# Preparing data for merging

All data have different granularity, so it need to be unified. I want to make granularity 1 hour, which would be best option for our task.

Key idea is to merge data by time and region id. Each row will have composed key created by `time` and `region_id` columns

## Alarms

In [7]:
alarms.head()

,region_id,region,time,alarm,has_started,has_ended
0,3,Хмельницька обл.,2022-02-26 16:00:00,1,1,0
1,3,Хмельницька обл.,2022-02-26 17:00:00,1,0,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1,1,0
3,3,Хмельницька обл.,2022-02-27 18:00:00,1,0,1
4,3,Хмельницька обл.,2022-02-27 19:00:00,1,1,0


In [8]:
alarms.info()

<class 'pandas.DataFrame'>
RangeIndex: 184656 entries, 0 to 184655
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   region_id    184656 non-null  int64
 1   region       184656 non-null  str  
 2   time         184656 non-null  str  
 3   alarm        184656 non-null  int64
 4   has_started  184656 non-null  int64
 5   has_ended    184656 non-null  int64
dtypes: int64(4), str(2)
memory usage: 16.8 MB


Convert `time` column to datetime format

In [9]:
alarms['time'] = pd.to_datetime(alarms['time'])

In [10]:
alarms

,region_id,region,time,alarm,has_started,has_ended
0,3,Хмельницька обл.,2022-02-26 16:00:00,1,1,0
1,3,Хмельницька обл.,2022-02-26 17:00:00,1,0,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1,1,0
3,3,Хмельницька обл.,2022-02-27 18:00:00,1,0,1
4,3,Хмельницька обл.,2022-02-27 19:00:00,1,1,0
...,...,...,...,...,...,...
184651,31,Київ,2026-04-03 08:00:00,1,0,1
184652,31,Київ,2026-04-03 12:00:00,1,1,1
184653,31,Київ,2026-04-04 03:00:00,1,1,0
184654,31,Київ,2026-04-04 04:00:00,1,0,1


In [11]:
# regions = alarms.groupby(['region_id', 'region_city']).size().reset_index().drop(columns=0)
# regions

In [12]:
alarms.isna().sum()

region_id      0
region         0
time           0
alarm          0
has_started    0
has_ended      0
dtype: int64

In [13]:
# alarms.start.min(), alarms.end.max()
alarms.time.min(), alarms.time.max()

(Timestamp('2022-02-24 07:00:00'), Timestamp('2026-04-07 13:00:00'))

Make 1 hour granularity

In [14]:
# # create a helper column with the list of hours per alarm
# from app.core.features.alarms_features import explode_by_hour, fix_regions

# alarms = fix_regions(alarms)
# alarms.head()

# alarm_expanded = explode_by_hour(alarms)
# alarm_expanded['time'] = pd.to_datetime(alarm_expanded['time'])
# alarm_expanded.head()

In [15]:
# alarm_expanded.info()

In [16]:
alarms.duplicated(subset=["region_id", "time"]).sum()

np.int64(0)

In [17]:
alarms = alarms.drop_duplicates(subset=["region_id", "time"])

In [18]:
alarms.shape

(184656, 6)

## Weather

In [19]:
weather.head()

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,0.0,0.0,82.64,-2.6,0.0,0.0,NaN,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,2022-02-24 00:00:00,Kropyvnytskyi
2,2.0,-1.1,80.51,-1.0,0.0,0.0,NaN,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,2022-02-24 00:00:00,Dnipro
3,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,2022-02-24 00:00:00,Kyiv
4,4.9,3.1,67.56,-0.6,0.0,0.0,NaN,7.6,29.4,1020.0,24.1,100.0,0.0,Overcast,2022-02-24 00:00:00,Kherson


In [20]:
weather.real_hour_datetime.min(), weather.real_hour_datetime.max()

('2022-02-24 00:00:00', '2026-04-08 23:00:00')

In [21]:
weather.isna().sum()

temp                       0
feelslike                  0
humidity                   0
dew                        0
precip                     0
precipprob                 0
preciptype            717975
windspeed                  0
winddir                    0
pressure                   0
visibility                 0
cloudcover                 0
uvindex                    0
conditions                 0
real_hour_datetime         0
city                       0
dtype: int64

In [22]:
weather.loc[weather.preciptype.isna()].sample(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
276642,17.6,17.6,79.82,14.1,0.0,0.0,NaN,10.8,300.0,1019.0,49.1,40.0,0.0,Partially cloudy,2023-07-10 06:00:00,Chernivtsi
183476,0.1,-3.9,95.04,-0.6,0.0,0.0,NaN,13.0,72.7,1031.0,0.0,66.9,1.0,Partially cloudy,2023-01-22 10:00:00,Kropyvnytskyi
359542,-4.5,-9.3,91.32,-5.7,0.0,0.0,NaN,11.9,103.5,1024.0,10.0,99.0,1.0,Overcast,2023-12-07 10:00:00,Khmelnytskyi


In [23]:
weather["preciptype"] = weather.preciptype.fillna("None")

In [24]:
weather.isna().sum()

temp                  0
feelslike             0
humidity              0
dew                   0
precip                0
precipprob            0
preciptype            0
windspeed             0
winddir               0
pressure              0
visibility            0
cloudcover            0
uvindex               0
conditions            0
real_hour_datetime    0
city                  0
dtype: int64

### Adding `region_id` column

In [25]:
weather_df = weather.copy()

In [26]:
from app.core.features.weather_features import add_region_ids

weather_df = add_region_ids(weather_df)

In [27]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [28]:
weather_df

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,region_id,time
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,Lutsk,8,2022-02-24 00:00:00
1,0.0,0.0,82.64,-2.6,0.0,0.0,None,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,Kropyvnytskyi,15,2022-02-24 00:00:00
2,2.0,-1.1,80.51,-1.0,0.0,0.0,None,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,Dnipro,9,2022-02-24 00:00:00
3,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,Kyiv,14,2022-02-24 00:00:00
4,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,Kyiv,31,2022-02-24 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
846810,4.6,0.3,76.91,0.9,0.1,80.6,['rain'],22.0,294.8,1018.0,24.1,99.9,0.0,"Rain, Overcast",Chernivtsi,26,2026-04-08 23:00:00
846811,4.2,1.5,71.99,-0.4,0.1,41.9,['rain'],10.8,294.2,1019.0,0.2,89.8,0.0,Partially cloudy,Lviv,27,2026-04-08 23:00:00
846812,3.8,3.8,89.93,2.3,0.0,35.5,['rain'],4.3,253.1,1006.0,24.1,32.0,0.0,Partially cloudy,Donetsk,28,2026-04-08 23:00:00
846813,3.9,0.9,87.41,2.0,0.3,29.0,['snow'],11.9,335.8,1012.0,0.1,100.0,0.0,Overcast,Kyiv,14,2026-04-08 23:00:00


## Telegram

In [29]:
telegram.head()

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1,2022-02-24 03:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
2,2022-02-24 04:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0
3,2022-02-24 05:00:00+02:00,17,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,19,3
4,2022-02-24 06:00:00+02:00,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,25,26,-3


In [30]:
telegram.info()

<class 'pandas.DataFrame'>
RangeIndex: 36083 entries, 0 to 36082
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   datetime                   36083 non-null  str  
 1   messages_count             36083 non-null  int64
 2   has_threat_sum             36083 non-null  int64
 3   nlp_артобстрілу            36083 non-null  int64
 4   nlp_бпла                   36083 non-null  int64
 5   nlp_відбій                 36083 non-null  int64
 6   nlp_відбій_тривоги         36083 non-null  int64
 7   nlp_дніпропетровська       36083 non-null  int64
 8   nlp_донецька               36083 non-null  int64
 9   nlp_запорізька             36083 non-null  int64
 10  nlp_нікополь               36083 non-null  int64
 11  nlp_нікополь_нікопольська  36083 non-null  int64
 12  nlp_нікопольська           36083 non-null  int64
 13  nlp_повітряна              36083 non-null  int64
 14  nlp_повітряна_тривога      36083 

In [31]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [32]:
tg_df.time.dtype

datetime64[us, Europe/Kyiv]

In [33]:
tg_df.duplicated(subset="time").sum()

np.int64(0)

## ISW

In [34]:
isw_raw.head()

,id,date,title,url,text
0,1745,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,"February 24, 3:00 pm EST Russian President Vla..."
1,1746,2022-02-24,Ukraine Conflict Update 6,https://understandingwar.org/research/russia-u...,"Institute for the Study of War, Russia Team 9:..."
2,1743,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Kateryna Stepa..."
3,1744,2022-02-25,Ukraine Conflict Update 7,https://understandingwar.org/research/russia-u...,Russia Team February 24 ISW published its most...
4,1741,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Katya Stepanen..."


In [35]:
isw_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1754 entries, 0 to 1753
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1754 non-null   int64
 1   date    1754 non-null   str  
 2   title   1754 non-null   str  
 3   url     1754 non-null   str  
 4   text    1754 non-null   str  
dtypes: int64(1), str(4)
memory usage: 49.2 MB


In [36]:
from app.core.features.isw_features import create_features_isw

isw = create_features_isw(isw_raw)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [37]:
isw.head()

,date,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,news_count_30d,news_velocity_30d,avg_dist_centroid_7d,news_count_7d,avg_dist_centroid_30d,anomaly_count_30d,anomaly_count_7d,topic_entropy_30d,centroid_shift_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_30d,dom_cluster_share_30d,topic_entropy_7d
0,2022-02-24,36261,0,0,0,2,0,0,0,0,0,0,0,0,NaN,0,NaN,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN
1,2022-02-25,31129,0,0,0,1,0,0,0,0,0,1,2,2,0.255692,2,0.255692,1,1,2.162327e-08,NaN,1.000000,2,NaN,1.000000,2.162327e-08
2,2022-02-26,37562,0,0,0,1,0,0,0,0,0,1,4,4,0.300419,4,0.300419,1,1,5.623352e-01,NaN,0.750000,4,NaN,0.750000,5.623352e-01
3,2022-02-27,54759,0,0,0,1,0,0,0,0,0,2,6,6,0.305603,6,0.305603,1,1,6.365142e-01,NaN,0.666667,6,NaN,0.666667,6.365142e-01
4,2022-02-28,43324,0,0,0,1,0,0,0,0,0,1,9,9,0.304577,9,0.304577,1,1,6.869616e-01,NaN,0.555556,9,NaN,0.555556,6.869616e-01


In [38]:
isw.date = pd.to_datetime(isw.date)

In [39]:
isw.date

0      2022-02-24
1      2022-02-25
2      2022-02-26
3      2022-02-27
4      2022-02-28
          ...    
1499   2026-04-03
1500   2026-04-04
1501   2026-04-05
1502   2026-04-06
1503   2026-04-07
Name: date, Length: 1504, dtype: datetime64[s]

In [40]:
isw.isna().sum()

date                      0
text_length               0
isw_cluster_0             0
isw_cluster_1             0
isw_cluster_2             0
isw_cluster_3             0
isw_cluster_4             0
isw_cluster_5             0
isw_cluster_6             0
isw_cluster_7             0
isw_cluster_8             0
isw_cluster_9             0
news_count_30d            0
news_velocity_30d         0
avg_dist_centroid_7d      1
news_count_7d             0
avg_dist_centroid_30d     1
anomaly_count_30d         0
anomaly_count_7d          0
topic_entropy_30d         1
centroid_shift_7d         8
dom_cluster_share_7d      1
news_velocity_7d          0
centroid_shift_30d       31
dom_cluster_share_30d     1
topic_entropy_7d          1
dtype: int64

In [41]:
isw.duplicated(subset="date").sum()

np.int64(0)

# Merging

## Creating spine

Idea is to create spine by multiplying hours by regions and then merge everythin to it by time and region_id

In [42]:
timeline = pd.date_range(
    alarms.time.min(),
    alarms.time.max(),
    freq="h"
)

region_ids = alarms.region_id.unique()

expected_length = len(timeline) * len(region_ids)

print(f"Number of hours: {len(timeline)}")
print(f"Number of regions: {len(region_ids)}")
print()
print(f"Expected length of spine: {expected_length}")

Number of hours: 36079
Number of regions: 24

Expected length of spine: 865896


In [43]:
spine = pd.MultiIndex.from_product([region_ids, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [44]:
spine.shape

(865896, 2)

In [45]:
spine.head()

,region_id,time
0,3,2022-02-24 07:00:00
1,3,2022-02-24 08:00:00
2,3,2022-02-24 09:00:00
3,3,2022-02-24 10:00:00
4,3,2022-02-24 11:00:00


In [46]:
spine.info()

<class 'pandas.DataFrame'>
RangeIndex: 865896 entries, 0 to 865895
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   region_id  865896 non-null  int64         
 1   time       865896 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1)
memory usage: 13.2 MB


## Merging alarms to spine

In [47]:
merged_df = spine.merge(alarms, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [48]:
merged_df

,region_id,time,region,alarm,has_started,has_ended
0,3,2022-02-24 07:00:00,NaN,0,NaN,NaN
1,3,2022-02-24 08:00:00,NaN,0,NaN,NaN
2,3,2022-02-24 09:00:00,NaN,0,NaN,NaN
3,3,2022-02-24 10:00:00,NaN,0,NaN,NaN
4,3,2022-02-24 11:00:00,NaN,0,NaN,NaN
...,...,...,...,...,...,...
865891,31,2026-04-07 09:00:00,NaN,0,NaN,NaN
865892,31,2026-04-07 10:00:00,NaN,0,NaN,NaN
865893,31,2026-04-07 11:00:00,NaN,0,NaN,NaN
865894,31,2026-04-07 12:00:00,NaN,0,NaN,NaN


## Merging weather

In [49]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [50]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 865896 entries, 0 to 865895
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   region_id    865896 non-null  int64         
 1   time         865896 non-null  datetime64[us]
 2   region       184656 non-null  str           
 3   alarm        865896 non-null  int64         
 4   has_started  184656 non-null  float64       
 5   has_ended    184656 non-null  float64       
 6   temp         845735 non-null  float64       
 7   feelslike    845735 non-null  float64       
 8   humidity     845735 non-null  float64       
 9   dew          845735 non-null  float64       
 10  precip       845735 non-null  float64       
 11  precipprob   845735 non-null  float64       
 12  preciptype   845735 non-null  str           
 13  windspeed    845735 non-null  float64       
 14  winddir      845735 non-null  float64       
 15  pressure     845735 non-null  float64       


## Merging telegram

In [51]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [52]:
tg_df.shape

(36083, 21)

In [53]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [54]:
merged_df.shape

(865896, 41)

## Merging ISW

In [55]:
isw.shape

(1504, 26)

In [56]:
merged_df["date"] = merged_df["time"].dt.date

In [57]:
merged_df.date = pd.to_datetime(merged_df.date)

In [58]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [59]:
merged_df = merged_df.drop(columns="date")

In [60]:
merged_df.shape

(865896, 66)

## Result

In [61]:
merged_df.head()

,region_id,time,region,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,news_count_30d,news_velocity_30d,avg_dist_centroid_7d,news_count_7d,avg_dist_centroid_30d,anomaly_count_30d,anomaly_count_7d,topic_entropy_30d,centroid_shift_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_30d,dom_cluster_share_30d,topic_entropy_7d
0,3,2022-02-24 07:00:00+02:00,NaN,0,NaN,NaN,1.2,-1.1,93.72,0.3,0.0,0.0,None,7.2,287.2,1024.0,0.5,100.0,0.0,Overcast,Khmelnytskyi,21.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,45.0,47.0,2.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,3,2022-02-24 08:00:00+02:00,NaN,0,NaN,NaN,1.4,-0.9,95.09,0.7,0.4,100.0,"['rain', 'snow']",7.2,280.0,1024.7,10.0,100.0,1.0,"Snow, Rain, Overcast",Khmelnytskyi,20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,67.0,0.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,3,2022-02-24 09:00:00+02:00,NaN,0,NaN,NaN,1.4,-1.4,91.05,0.1,0.0,0.0,None,9.0,295.3,1025.0,23.7,88.5,0.0,Partially cloudy,Khmelnytskyi,19.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86.0,-1.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,3,2022-02-24 10:00:00+02:00,NaN,0,NaN,NaN,2.0,-0.6,85.35,-0.2,0.0,0.0,None,8.6,311.1,1026.0,23.5,88.7,1.0,Partially cloudy,Khmelnytskyi,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0,99.0,0.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,3,2022-02-24 11:00:00+02:00,NaN,0,NaN,NaN,2.2,-0.9,96.49,1.7,0.0,0.0,None,10.8,320.0,1025.8,10.0,100.0,1.0,Overcast,Khmelnytskyi,22.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0,121.0,0.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [62]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 865896 entries, 0 to 865895
Data columns (total 66 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  865896 non-null  int64                      
 1   time                       865680 non-null  datetime64[us, Europe/Kyiv]
 2   region                     184656 non-null  str                        
 3   alarm                      865896 non-null  int64                      
 4   has_started                184656 non-null  float64                    
 5   has_ended                  184656 non-null  float64                    
 6   temp                       845735 non-null  float64                    
 7   feelslike                  845735 non-null  float64                    
 8   humidity                   845735 non-null  float64                    
 9   dew                        845735 non-null  floa

In [63]:
merged_df.shape

(865896, 66)

In [64]:
print(f"Expected length: {expected_length}")
print(f"Actual length: {merged_df.shape[0]}")

Expected length: 865896
Actual length: 865896


In [65]:
merged_df.region_id.nunique()

24

In [66]:
merged_df.city.nunique()

23

In [67]:
merged_df.region.nunique()

24

# Preprocessing merged data

In [68]:
df = merged_df.copy()

In [69]:
df = df.loc[~df.time.isna()]

In [70]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
region                       681078
alarm                             0
has_started                  681078
has_ended                    681078
temp                          20040
feelslike                     20040
humidity                      20040
dew                           20040
precip                        20040
precipprob                    20040
preciptype                    20040
windspeed                     20040
winddir                       20040
pressure                      20040
visibility                    20040
cloudcover                    20040
uvindex                       20040
conditions                    20040
city                          20040
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська        

In [71]:
df = df.drop(["city", "region"], axis=1) # region_id column already represents this information

In [72]:
df.loc[df.temp.isna(), "time"].agg(['max', 'min', 'median'])

max      2026-04-06 23:00:00+03:00
min      2022-07-03 21:00:00+03:00
median   2026-03-18 13:00:00+02:00
Name: time, dtype: datetime64[us, Europe/Kyiv]

In [73]:
df = df.loc[~df.temp.isna()]

df.text_length = df.text_length.fillna(-1)
df.preciptype = df.preciptype.fillna("None")
df = df.fillna({"has_started": 0, "has_ended": 0})

isw_cluster_cols = [col for col in df.columns if col.startswith("isw_cluster")]
df[isw_cluster_cols] = df[isw_cluster_cols].fillna(0)

df.visibility = df.visibility.ffill()
df.uvindex = df.uvindex.ffill()

df = df.loc[~df.messages_count.isna()]

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour
df['hour_of_day'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [74]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                        0
time                             0
alarm                            0
has_started                      0
has_ended                        0
temp                             0
feelslike                        0
humidity                         0
dew                              0
precip                           0
precipprob                       0
preciptype                       0
windspeed                        0
winddir                          0
pressure                         0
visibility                       0
cloudcover                       0
uvindex                          0
conditions                       0
messages_count                   0
has_threat_sum                   0
nlp_артобстрілу                  0
nlp_бпла                         0
nlp_відбій                       0
nlp_відбій_тривоги               0
nlp_дніпропетровська             0
nlp_донецька                     0
nlp_запорізька                   0
nlp_нікополь        

In [75]:
df.loc[df.centroid_shift_30d.isna(), "time"].agg(['min', 'max', 'median'])

min      2022-02-24 07:00:00+02:00
max      2022-03-26 23:00:00+02:00
median   2022-03-11 15:00:00+02:00
Name: time, dtype: datetime64[us, Europe/Kyiv]

missing values only at first month of war, so we can drop them.

In [76]:
df = df.dropna(axis=0)

In [77]:
df.isna().sum().sum()

np.int64(0)

In [78]:
df = df.reset_index(drop=True)

In [79]:
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

cat_cols = list(df.select_dtypes(include=["object", "category"]).columns)

df[cat_cols] = encoder.fit_transform(df[cat_cols])

In [80]:
df.head(3)

,region_id,time,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,news_count_30d,news_velocity_30d,avg_dist_centroid_7d,news_count_7d,avg_dist_centroid_30d,anomaly_count_30d,anomaly_count_7d,topic_entropy_30d,centroid_shift_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_30d,dom_cluster_share_30d,topic_entropy_7d,year,month,day,hour,hour_of_day,day_of_week,is_weekend
0,3,2022-03-27 00:00:00+02:00,0,0.0,0.0,7.0,3.1,68.99,1.7,0.0,0.0,0.0,24.5,312.8,1018.0,24.1,55.8,0.0,4.0,12.0,8.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,8.0,8.0,8.0,0.0,2.0,37.0,360.0,1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.7,2.0,0.533511,0.62,0.610864,2022,3,27,0,0,6,1
1,3,2022-03-27 01:00:00+02:00,0,0.0,0.0,5.7,1.2,44.72,-5.4,0.0,0.0,0.0,26.6,326.8,1019.0,24.1,4.2,0.0,0.0,23.0,10.0,0.0,0.0,11.0,11.0,2.0,0.0,2.0,0.0,0.0,0.0,10.0,10.0,12.0,11.0,2.0,54.0,378.0,2.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.7,2.0,0.533511,0.62,0.610864,2022,3,27,1,1,6,1
2,3,2022-03-27 02:00:00+02:00,0,0.0,0.0,3.8,-0.9,43.48,-7.5,0.0,0.0,0.0,23.8,322.6,1021.0,24.1,0.0,0.0,0.0,5.0,1.0,0.0,0.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,40.0,382.0,-9.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.7,2.0,0.533511,0.62,0.610864,2022,3,27,2,2,6,1


In [81]:
df.sort_values(by=["region_id", "time"])

,region_id,time,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,news_count_30d,news_velocity_30d,avg_dist_centroid_7d,news_count_7d,avg_dist_centroid_30d,anomaly_count_30d,anomaly_count_7d,topic_entropy_30d,centroid_shift_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_30d,dom_cluster_share_30d,topic_entropy_7d,year,month,day,hour,hour_of_day,day_of_week,is_weekend
0,3,2022-03-27 00:00:00+02:00,0,0.0,0.0,7.0,3.1,68.99,1.7,0.0,0.0,0.0,24.5,312.8,1018.0,24.1,55.8,0.0,4.0,12.0,8.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,8.0,8.0,8.0,0.0,2.0,37.0,360.0,1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.700,2.0,0.533511,0.620000,0.610864,2022,3,27,0,0,6,1
1,3,2022-03-27 01:00:00+02:00,0,0.0,0.0,5.7,1.2,44.72,-5.4,0.0,0.0,0.0,26.6,326.8,1019.0,24.1,4.2,0.0,0.0,23.0,10.0,0.0,0.0,11.0,11.0,2.0,0.0,2.0,0.0,0.0,0.0,10.0,10.0,12.0,11.0,2.0,54.0,378.0,2.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.700,2.0,0.533511,0.620000,0.610864,2022,3,27,1,1,6,1
2,3,2022-03-27 02:00:00+02:00,0,0.0,0.0,3.8,-0.9,43.48,-7.5,0.0,0.0,0.0,23.8,322.6,1021.0,24.1,0.0,0.0,0.0,5.0,1.0,0.0,0.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,40.0,382.0,-9.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.700,2.0,0.533511,0.620000,0.610864,2022,3,27,2,2,6,1
3,3,2022-03-27 04:00:00+03:00,0,0.0,0.0,2.2,-2.9,47.95,-7.7,0.0,0.0,0.0,23.0,320.1,1021.0,24.1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.0,382.0,-1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.700,2.0,0.533511,0.620000,0.610864,2022,3,27,4,4,6,1
4,3,2022-03-27 05:00:00+03:00,0,0.0,0.0,1.1,-4.3,52.28,-7.6,0.0,0.0,0.0,22.7,319.1,1022.0,24.1,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,7.0,383.0,1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,48.0,0.388121,10.0,0.471807,5.0,1.0,0.664064,0.173424,0.700,2.0,0.533511,0.620000,0.610864,2022,3,27,5,5,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827947,31,2026-04-07 09:00:00+03:00,0,0.0,0.0,4.4,0.9,71.51,-0.3,0.0,0.0,0.0,15.1,264.8,1009.0,24.1,87.2,2.0,4.0,32.0,1.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,16.0,16.0,16.0,12.0,12.0,14.0,2.0,2.0,75.0,523.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.0,0.0,0.277971,8.0,0.270821,0.0,0.0,0.362211,0.165876,0.875,0.0,0.282359,0.882353,0.376770,2026,4,7,9,9,1,0
827948,31,2026-04-07 10:00:00+03:00,0,0.0,0.0,6.1,2.4,62.62,-0.5,0.0,0.0,0.0,20.2,290.7,1008.0,24.1,98.5,3.0,3.0,46.0,4.0,3.0,0.0,21.0,19.0,0.0,0.0,0.0,10.0,10.0,10.0,9.0,9.0,12.0,19.0,2.0,46.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.0,0.0,0.277971,8.0,0.270821,0.0,0.0,0.362211,0.165876,0.875,0.0,0.282359,0.882353,0.376770,2026,4,7,10,10,1,0
827949,31,2026-04-07 11:00:00+03:00,0,0.0,0.0,6.1,2.6,67.33,0.5,0.2,100.0,4.0,18.0,287.1,1008.0,1.6,100.0,3.0,7.0,16.0,3.0,2.0,0.0,3.0,2.0,0.0,0.0,0.0,5.0,5.0,5.0,5.0,5.0,5.0,2.0,0.0,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.0,0.0,0.277971,8.0,0.270821,0.0

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 827952 entries, 0 to 827951
Data columns (total 71 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  827952 non-null  int64                      
 1   time                       827952 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      827952 non-null  int64                      
 3   has_started                827952 non-null  float64                    
 4   has_ended                  827952 non-null  float64                    
 5   temp                       827952 non-null  float64                    
 6   feelslike                  827952 non-null  float64                    
 7   humidity                   827952 non-null  float64                    
 8   dew                        827952 non-null  float64                    
 9   precip                     827952 non-null  floa

In [83]:
cols_to_int = ['messages_count', 'has_threat_sum',
       'nlp_артобстрілу', 'nlp_бпла', 'nlp_відбій', 'nlp_відбій_тривоги',
       'nlp_дніпропетровська', 'nlp_донецька', 'nlp_запорізька',
       'nlp_нікополь', 'nlp_нікополь_нікопольська', 'nlp_нікопольська',
       'nlp_повітряна', 'nlp_повітряна_тривога', 'nlp_тривога', 'nlp_тривоги',
       'nlp_харківська', 'msg_count_last_3h', 'msg_count_last_24h',
       'threat_diff_1h', 'day_of_week', 'is_weekend',
       'text_length', 'isw_cluster_0', 'isw_cluster_1', 'isw_cluster_2',
       'isw_cluster_3', 'isw_cluster_4', 'isw_cluster_5', 'isw_cluster_6',
       'isw_cluster_7', 'isw_cluster_8', 'isw_cluster_9', "news_count_7d", 
        "news_count_30d", "anomaly_count_7d", "anomaly_count_30d",
        "news_velocity_7d", "news_velocity_30d"]

df[cols_to_int] = df[cols_to_int].astype(int)

In [84]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 827952 entries, 0 to 827951
Data columns (total 71 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  827952 non-null  int64                      
 1   time                       827952 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      827952 non-null  int64                      
 3   has_started                827952 non-null  float64                    
 4   has_ended                  827952 non-null  float64                    
 5   temp                       827952 non-null  float64                    
 6   feelslike                  827952 non-null  float64                    
 7   humidity                   827952 non-null  float64                    
 8   dew                        827952 non-null  float64                    
 9   precip                     827952 non-null  floa

In [85]:
df.region_id.unique()

array([ 3,  4,  5,  8,  9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22,
       23, 24, 25, 26, 27, 28, 31])

# Saving

In [86]:
save_path = Path("data/merged")

df.to_csv(save_path / "merged_v7.csv", index=False)

# Extracting features

Calculate distance in regions to nearest region with alarm

In [87]:
# %%time
# from collections import deque


# NEIGHBORS = {
#     3:  [4, 5, 10, 21, 27],        # Khmelnytska: Vinnytska, Rivnenska, Zhytomyrska, Ternopilska, Lvivska
#     4:  [3, 10, 14, 15, 18, 24, 26], # Vinnytska: Khmelnytska, Zhytomyrska, Kyivska, Kirovohradska, Odeska, Cherkaska, Chernivetska
#     5:  [3, 8, 10, 21, 27],        # Rivnenska: Khmelnytska, Volynska, Zhytomyrska, Ternopilska, Lvivska
#     8:  [5, 10, 27],               # Volynska: Rivnenska, Zhytomyrska, Lvivska
#     9:  [12, 15, 19, 22, 24, 28],  # Dnipropetrovska: Zaporizka, Kirovohradska, Poltavska, Kharkivska, Cherkaska, Donetska
#     10: [3, 4, 5, 8, 14, 25, 31],  # Zhytomyrska: Khmelnytska, Vinnytska, Rivnenska, Volynska, Kyivska, Chernihivska, Kyiv
#     11: [13, 27],                  # Zakarpatska: Ivano-Frankivska, Lvivska
#     12: [9, 15, 17, 23, 28],       # Zaporizka: Dnipropetrovska, Kirovohradska, Mykolaivska, Khersonska, Donetska
#     13: [11, 21, 26, 27],          # Ivano-Frankivska: Zakarpatska, Ternopilska, Chernivetska, Lvivska
#     14: [4, 10, 24, 25, 31],       # Kyivska: Vinnytska, Zhytomyrska, Cherkaska, Chernihivska, Kyiv
#     15: [4, 9, 12, 17, 18, 24],    # Kirovohradska: Vinnytska, Dnipropetrovska, Zaporizka, Mykolaivska, Odeska, Cherkaska
#     16: [22, 28],                  # Luhanska: Kharkivska, Donetska
#     17: [12, 15, 18, 23],          # Mykolaivska: Zaporizka, Kirovohradska, Odeska, Khersonska
#     18: [4, 15, 17, 26],           # Odeska: Vinnytska, Kirovohradska, Mykolaivska, Chernivetska
#     19: [9, 20, 22, 24, 25],       # Poltavska: Dnipropetrovska, Sumska, Kharkivska, Cherkaska, Chernihivska
#     20: [19, 22, 25],              # Sumska: Poltavska, Kharkivska, Chernihivska
#     21: [3, 5, 13, 26, 27],        # Ternopilska: Khmelnytska, Rivnenska, Ivano-Frankivska, Chernivetska, Lvivska
#     22: [9, 16, 19, 20, 28],       # Kharkivska: Dnipropetrovska, Luhanska, Poltavska, Sumska, Donetska
#     23: [12, 17],                  # Khersonska: Zaporizka, Mykolaivska
#     24: [4, 9, 14, 15, 19, 25, 31], # Cherkaska: Vinnytska, Dnipropetrovska, Kyivska, Kirovohradska, Poltavska, Chernihivska, Kyiv
#     25: [10, 14, 19, 20, 24, 31],  # Chernihivska: Zhytomyrska, Kyivska, Poltavska, Sumska, Cherkaska, Kyiv
#     26: [4, 13, 18, 21],           # Chernivetska: Vinnytska, Ivano-Frankivska, Odeska, Ternopilska
#     27: [3, 5, 8, 11, 13, 21],     # Lvivska: Khmelnytska, Rivnenska, Volynska, Zakarpatska, Ivano-Frankivska, Ternopilska
#     28: [9, 12, 16, 22],           # Donetska: Dnipropetrovska, Zaporizka, Luhanska, Kharkivska
#     31: [10, 14, 24, 25],          # Kyiv (city): Zhytomyrska, Kyivska, Cherkaska, Chernihivska
# }

# # Precompute BFS distances between all region pairs once at import time
# def _bfs_distances(source, neighbors_map):
#     """Returns dict of {region_id: hops} from source via BFS."""
#     dist = {source: 0}
#     queue = deque([source])
#     while queue:
#         node = queue.popleft()
#         for neighbor in neighbors_map[node]:
#             if neighbor not in dist:
#                 dist[neighbor] = dist[node] + 1
#                 queue.append(neighbor)
#     return dist

# ALL_GRAPH_DISTANCES = {
#     r: _bfs_distances(r, NEIGHBORS)
#     for r in NEIGHBORS
# }

# def dist_to_nearest_alarm(row, alarm_lookup, region_col="region_id", time_col="time", alarm_col="alarm"):
#     """
#     For use with df.apply(). Returns:
#       0    — current region has an alarm
#       hops — min number of borders to cross to reach a region with alarm
#       -1   — no alarms anywhere at this timestamp
    
#     Parameters:
#         row          - DataFrame row (from apply axis=1)
#         alarm_lookup - dict mapping (timestamp, region_id) -> alarm value;
#                        build it once before calling apply (see usage below)
#         region_col   - name of the region ID column
#         time_col     - name of the timestamp column
#         alarm_col    - name of the alarm status column
#     """
#     current_region = row[region_col]
#     current_time   = row[time_col]

#     # Current region has alarm → distance is 0
#     if row[alarm_col]:
#         return 0

#     distances_from_current = ALL_GRAPH_DISTANCES[current_region]

#     min_hops = float("inf")
#     for region_id in NEIGHBORS:
#         if region_id == current_region:
#             continue
#         try:
#             has_alarm = alarm_lookup[(current_time, region_id)]
#         except KeyError:
#             continue
#         if has_alarm:
#             hops = distances_from_current.get(region_id, float("inf"))
#             if hops < min_hops:
#                 min_hops = hops

#     return -1 if min_hops == float("inf") else min_hops


# # --- Usage ---

# # Build lookup ONCE before apply (much faster than indexing df inside apply)
# alarm_lookup = df.set_index(["time", "region_id"])["alarm"].to_dict()

# df["hops_to_nearest_alarm"] = df.apply(
#     dist_to_nearest_alarm,
#     axis=1,
#     alarm_lookup=alarm_lookup,
#     region_col="region_id",
#     time_col="time",
#     alarm_col="alarm",
# )

# df["hops_to_nearest_alarm"] = df.groupby("region_id")["hops_to_nearest_alarm"].shift(1)

In [88]:
# df.hops_to_nearest_alarm.unique()

Calculate number of alarms in all regions by hour

In [89]:
all_alarms = df.groupby("time")["alarm"].sum().rename("num_alarms").reset_index()

df = pd.merge(df, all_alarms, on="time", how="left")
# df["other_alarms_count"] = df["num_alarms"] - df["alarm"]
for i in range(12):
    df[f"alarms_count_{i+1}h_ago"] = df["num_alarms"].shift(i+1)

df = df.drop(columns="num_alarms")

Add lag for 1-24 hours

In [90]:
for i in range(24):
    df[f"alarm_status_{i+1}h_ago"] = df.groupby("region_id")["alarm"].shift(i+1)

In [91]:
df = df.dropna(axis=0).reset_index(drop=True)

In [92]:
df.head()

,region_id,time,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,news_count_30d,news_velocity_30d,avg_dist_centroid_7d,news_count_7d,avg_dist_centroid_30d,anomaly_count_30d,anomaly_count_7d,topic_entropy_30d,centroid_shift_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_30d,dom_cluster_share_30d,topic_entropy_7d,year,month,day,hour,hour_of_day,day_of_week,is_weekend,alarms_count_1h_ago,alarms_count_2h_ago,alarms_count_3h_ago,alarms_count_4h_ago,alarms_count_5h_ago,alarms_count_6h_ago,alarms_count_7h_ago,alarms_count_8h_ago,alarms_count_9h_ago,alarms_count_10h_ago,alarms_count_11h_ago,alarms_count_12h_ago,alarm_status_1h_ago,alarm_status_2h_ago,alarm_status_3h_ago,alarm_status_4h_ago,alarm_status_5h_ago,alarm_status_6h_ago,alarm_status_7h_ago,alarm_status_8h_ago,alarm_status_9h_ago,alarm_status_10h_ago,alarm_status_11h_ago,alarm_status_12h_ago,alarm_status_13h_ago,alarm_status_14h_ago,alarm_status_15h_ago,alarm_status_16h_ago,alarm_status_17h_ago,alarm_status_18h_ago,alarm_status_19h_ago,alarm_status_20h_ago,alarm_status_21h_ago,alarm_status_22h_ago,alarm_status_23h_ago,alarm_status_24h_ago
0,3,2022-03-28 01:00:00+03:00,0,0.0,0.0,-0.1,-1.8,48.02,-9.8,0.0,0.0,0.0,5.0,291.3,1030.0,24.1,1.6,0.0,0.0,8,2,0,0,3,3,0,0,0,0,0,0,1,1,1,4,0,45,285,-3,13953,0,0,0,0,0,0,0,0,0,1,49,45,0.386532,10,0.465249,5,1,0.657529,0.179787,0.7,2,0.484262,0.632653,0.610864,2022,3,28,1,1,0,0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,2022-03-28 02:00:00+03:00,0,0.0,0.0,-0.6,-0.6,51.39,-9.4,0.0,0.0,0.0,4.7,271.0,1030.0,24.1,2.5,0.0,0.0,7,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,28,269,-2,13953,0,0,0,0,0,0,0,0,0,1,49,45,0.386532,10,0.465249,5,1,0.657529,0.179787,0.7,2,0.484262,0.632653,0.610864,2022,3,28,2,2,0,0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2022-03-28 03:00:00+03:00,0,0.0,0.0,-0.8,-4.5,59.99,-7.6,0.0,0.0,0.0,10.8,270.0,1029.4,10.0,40.0,0.0,4.0,11,10,0,0,0,0,2,0,2,0,0,0,9,9,9,0,2,26,275,10,13953,0,0,0,0,0,0,0,0,0,1,49,45,0.386532,10,0.465249,5,1,0.657529,0.179787,0.7,2,0.484262,0.632653,0.610864,2022,3,28,3,3,0,0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2022-03-28 04:00:00+03:00,0,0.0,0.0,-1.3,-1.3,56.25,-8.9,0.0,0.0,0.0,4.3,228.8,1029.0,24.1,18.8,0.0,0.0,9,1,0,0,6,6,2,0,2,0,0,0,1,1,2,6,2,27,283,-9,13953,0,0,0,0,0,0,0,0,0,1,49,45,0.386532,10,0.465249,5,1,0.657529,0.179787,0.7,2,0.484262,0.632653,0.610864,2022,3,28,4,4,0,0,8.0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3,2022-03-28 05:00:00+03:00,0,0.0,0.0,-1.3,-1.3,56.69,-8.8,0.0,0.0,0.0,3.6,199.3,1029.0,24.1,87.8,0.0,4.0,4,1,0,0,3,2,0,0,0,0,0,0,1,1,1,3,2,24,286,0,13953,0,0,0,0,0,0,0,0,0,1,49,45,0.386532,10,0.465249,5,1,0.657529,0.179787,0.7,2,0.484262,0.632653,0.610864,2022,3,28,5,5,0,0,9.0,8.0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Saving result

In [93]:
df.to_csv(save_path / "merged_preprocessed.csv", index=False)

In [94]:
import joblib


model_path = Path("app/models/preprocessing")
if not model_path.exists():
    model_path.mkdir(parents=True, exist_ok=True)

joblib.dump(encoder, model_path / "merged_df_encoder.joblib")

['app\\models\\preprocessing\\merged_df_encoder.joblib']